In [ ]:
%conda install mysql-connector-python python-dotenv pandas # 라이브러리 세팅

In [10]:
import mysql.connector as mc # 필요한 라이브러리 import 한다.
from mysql.connector import Error
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import pandas as pd

In [12]:
load_dotenv() # 환경변수 읽는 것
DB_CONFIG = {
    'host' : os.getenv('DB_HOST'),
    'user' : os.getenv('DB_USER'),
    'password' : os.getenv('DB_PASSWORD'),
    'database' : os.getenv('DB_DATABASE'),
    'port' : os.getenv('DB_PORT')
}

In [20]:
class ShopDB:
    '''
    쇼핑몰 데이터베이스 관리 클래스     
    '''
    def __init__(self,host,user,password,database,port=3306): # 생성자에 DB 정보 넣기
        self.host = host
        self.user = user
        self.password = password
        self.database = database
        self.port = port
    def connect(self):
        '''연결''' 
        try:
            self.conn = mc.connect(host=self.host, user=self.user,
                    password = self.password, database=self.database,
                    port=self.port) # 연결 할 떄 host,user,password,database,port를 가져온다.
            self.cursor = self.conn.cursor() # 데이터베이스 열어서 실행 리모컨 역할
            self.cursor.execute(f'use {self.database}') # 데이터베이스를 실행한다.
            return True  #연결 완료
        except Error as e: # 연결에서 오류가 나면 에러를 보여줌
            print(f'데이터베이스 에러 : {e}')
            return False #연결 실패
    def close(self):
        '''연결종료'''
        if self.cursor:
            self.cursor.close()
        if self.conn and self.conn.is_connected(): # 현재 연결 객체가 존재하고 연결되어있는 것인지 확인한다.
            self.conn.close()
        print('연결 종료')
    def execute(self,query,params = None):
        '''쿼리실행'''
        try:
            if params: # 전달할 데이터가 있으면
                self.cursor.execute(query,params)
            else:
                self.cursor.execute(query)
        except Error as e:
            print(f'쿼리 에러 : {e}')
    def fetchall(self):
        return self.cursor.fetchall()
    def fetchone(self):
        return self.cursor.fetchone()
    def query_to_dataframe(self,query,params = None):
        ''''쿼리결과를 pandas dataframe으로 변환'''
        self.execute(query,params)
        columns = [desc[0] for desc in self.cursor.description] # 모든 컬럼에 대해 첫번째 요소만 뽑아서 리스트로 만든다(리스트 컴프리헨션)
        data = self.cursor.fetchall()
        return pd.DataFrame(data,columns=columns)
    def commit(self):
        if self.conn:
            self.conn.commit()

# 연결확인

In [14]:
sh = ShopDB(**DB_CONFIG)
sh.connect()
sh.close()

연결 종료


# 테이블 생성 
- 숙제(실습은 테이블 수동생성)

In [ ]:
# 테이블 삭제(초기화)
db = ShopDB(**DB_CONFIG)
db.connect()

tables = ['order_details', 'orders', 'customers', 'products']

for table in tables:
    db.execute(f'drop table if exists {table}')

db.commit()

In [21]:
db = ShopDB(**DB_CONFIG)
db.connect()

# customer 테이블 생성
with open('query.txt', 'r', encoding='utf-8') as f:    
   query = f.read()

db.execute(query)
db.commit()
db.close()

쿼리 에러 : 1050 (42S01): Table 'customers' already exists
연결 종료


In [ ]:
db = ShopDB(**DB_CONFIG)
db.connect()

customers_data = [
    ('김철수', 'kim@email.com', 'Gold', 5000, '2023-01-15'),
    ('이영희', 'lee@email.com', 'Silver', 3000, '2023-03-20'),
    ('박민수', 'park@email.com', 'Bronze', 1000, '2023-06-10'),
    ('최지은', 'choi@email.com', 'Gold', 6000, '2023-02-05'),
    ('정수진', 'jung@email.com', 'Silver', 2500, '2023-04-18'),
    ('강동원', 'kang@email.com', 'Bronze', 500, '2024-01-20'),
    ('윤서연', 'yoon@email.com', 'Silver', 3500, '2023-05-12'),
    ('임하늘', 'lim@email.com', 'Bronze', 800, '2024-02-01'),
    ('송민호', 'song@email.com', 'Gold', 7000, '2023-01-28'),
    ('한지민', 'han@email.com', 'Bronze', 1200, '2023-07-15')
]
query = '''insert into customers (name, email, grade,
    point,join_date) values(%s,%s,%s,%s,%s)'''
for customer in customers_data:
    db.execute(query,customer)
db.commit()
db.close()

연결 종료


In [ ]:
# 상품 테이블
db = ShopDB(**DB_CONFIG)
db.connect()

products_data = [
    ('노트북', '전자기기', 1200000, 15),
    ('무선마우스', '전자기기', 35000, 50),
    ('키보드', '전자기기', 89000, 30),
    ('모니터', '전자기기', 350000, 20),
    ('청바지', '의류', 59000, 100),
    ('티셔츠', '의류', 25000, 150),
    ('운동화', '의류', 89000, 80),
    ('백팩', '가방', 45000, 40),
    ('텀블러', '생활용품', 15000, 200),
    ('책상스탠드', '생활용품', 28000, 60),
] 
query = '''insert into products (product_name, category, price,
    stock) values(%s,%s,%s,%s)'''
for data in products_data:
    db.execute(query,data)
db.commit()
db.close()


연결 종료


In [ ]:
# 주문 테이블
db = ShopDB(**DB_CONFIG)
db.connect()

orders_data = [
    (1, '2024-02-15 10:30:00', 1235000, 'Completed'),
    (1, '2024-02-20 14:20:00', 89000, 'Completed'),
    (2, '2024-02-16 09:15:00', 375000, 'Completed'),
    (3, '2024-02-17 16:45:00', 143000, 'Completed'),
    (4, '2024-02-18 11:00:00', 1550000, 'Completed'),
    (5, '2024-02-19 13:30:00', 84000, 'Completed'),
    (6, '2024-02-21 10:00:00', 59000, 'Pending'),
    (7, '2024-02-22 15:20:00', 254000, 'Completed'),
    (8, '2024-02-23 12:10:00', 45000, 'Cancelled'),
    (9, '2024-02-24 14:50:00', 1289000, 'Completed'),
]
query = '''insert into orders (customer_id, order_date, total_amount,
    status) values(%s,%s,%s,%s)'''
for data in orders_data:
    db.execute(query,data)
db.commit()
db.close()


연결 종료


In [ ]:
# order_detail 테이블 
db = ShopDB(**DB_CONFIG)
db.connect()

order_details_data = [
    (1, 1, 1, 1200000, 1200000),
    (1, 2, 1, 35000, 35000),
    (2, 3, 1, 89000, 89000),
    (3, 4, 1, 350000, 350000),
    (3, 5, 1, 25000, 25000),
    (4, 5, 2, 59000, 118000),
    (4, 6, 1, 25000, 25000),
    (5, 1, 1, 1200000, 1200000),
    (5, 4, 1, 350000, 350000),
    (6, 7, 1, 89000, 89000),
    (7, 5, 1, 59000, 59000),
    (8, 2, 3, 35000, 105000),
    (8, 3, 1, 89000, 89000),
    (8, 9, 2, 15000, 30000),
    (8, 10, 1, 28000, 28000),
    (9, 8, 1, 45000, 45000),
    (10, 1, 1, 1200000, 1200000),
    (10, 3, 1, 89000, 89000),
]
query = '''insert into order_details (order_id, product_id, quantity,
    unit_price,subtotal) values(%s,%s,%s,%s,%s)'''
for data in order_details_data:
    db.execute(query,data)
db.commit()
db.close()


연결 종료


# 데이터 조회(select)

In [42]:
db = ShopDB(**DB_CONFIG)
db.connect()

df_customers = db.query_to_dataframe('select * from customers')

db.close()

연결 종료


In [43]:
df_customers

,customer_id,name,email,phone,point,grade,join_date,last_purchase_date,created_at
0,1,김철수,kim@email.com,None,5000,Gold,2023-01-15,None,2026-03-25 10:29:51
1,2,이영희,lee@email.com,None,3000,Silver,2023-03-20,None,2026-03-25 10:29:51
2,3,박민수,park@email.com,None,1000,Bronze,2023-06-10,None,2026-03-25 10:29:51
3,4,최지은,choi@email.com,None,6000,Gold,2023-02-05,None,2026-03-25 10:29:51
4,5,정수진,jung@email.com,None,2500,Silver,2023-04-18,None,2026-03-25 10:29:51
5,6,강동원,kang@email.com,None,500,Bronze,2024-01-20,None,2026-03-25 10:29:51
6,7,윤서연,yoon@email.com,None,3500,Silver,2023-05-12,None,2026-03-25 10:29:51
7,8,임하늘,lim@email.com,None,800,Bronze,2024-02-01,None,2026-03-25 10:29:51
8,9,송민호,song@email.com,None,7000,Gold,2023-01-28,None,2026-03-25 10:29:51
9,10,한지민,han@email.com,None,1200,Bronze,2023-07-15,None,2026-03-25 10:29:51
